# Onvo — راه‌اندازی خودکار

این نوت‌بوک همهٔ کارها را خودش انجام می‌دهد:

1. سورس را در ریپوی تو آپلود می‌کند (شامل `.github` که آپلودکنندهٔ وب معمولاً می‌بلعد)
2. کلید امضا می‌سازد
3. چهار secret را تنظیم می‌کند
4. ساخت را اجرا و تا آخر دنبال می‌کند
5. APK را برایت دانلود می‌کند

---

### دربارهٔ امنیت توکن

توکن با `getpass` گرفته می‌شود، یعنی **روی صفحه دیده نمی‌شود و در خروجی
نوت‌بوک ذخیره نمی‌ماند**. فقط در حافظهٔ همین جلسه می‌ماند و با بستن
نوت‌بوک از بین می‌رود.

توکن را هیچ‌جای دیگری ننویس — نه در چت، نه در سلول کد.

بعد از تمام شدن کار، توکن را از
[github.com/settings/tokens](https://github.com/settings/tokens) حذف کن.

## گام ۱ — ساخت توکن

روی گوشی این لینک را باز کن:

**[github.com/settings/tokens/new](https://github.com/settings/tokens/new)**

تنظیمات:

| فیلد | مقدار |
|---|---|
| Note | `onvo-setup` |
| Expiration | **7 days** |
| Scope | فقط تیک **`repo`** و **`workflow`** |

> این توکن کلاسیک است. حتماً هر دو تیک `repo` و `workflow` را بزن —
> بدون `workflow` گیت‌هاب اجازهٔ آپلود پوشهٔ `.github` را نمی‌دهد.

**Generate token** را بزن و رشتهٔ `ghp_...` را کپی کن.

In [ ]:
#@title گام ۲ — ورود اطلاعات { display-mode: "form" }
import getpass, re, sys, json, base64, time, io, os, zipfile
import urllib.request, urllib.error

GITHUB_USER = "parvaneshahraki978-coder"  #@param {type:"string"}
REPO_NAME   = "Onvo"                      #@param {type:"string"}

TOKEN = getpass.getpass("توکن گیت‌هاب را پیست کن (دیده نمی‌شود): ").strip()

if not TOKEN:
    sys.exit("توکن وارد نشد.")
if not re.match(r'^(ghp_|github_pat_)', TOKEN):
    print("⚠️  این شبیه توکن گیت‌هاب نیست، ولی ادامه می‌دهم.")

API  = "https://api.github.com"
REPO = f"{GITHUB_USER}/{REPO_NAME}"

def gh(method, path, body=None, raw=False):
    url = path if path.startswith("http") else f"{API}{path}"
    data = json.dumps(body).encode() if body is not None else None
    req  = urllib.request.Request(url, data=data, method=method)
    req.add_header("Authorization", f"Bearer {TOKEN}")
    req.add_header("Accept", "application/vnd.github+json")
    req.add_header("X-GitHub-Api-Version", "2022-11-28")
    if data: req.add_header("Content-Type", "application/json")
    try:
        with urllib.request.urlopen(req) as r:
            payload = r.read()
            return r.status, (payload if raw else (json.loads(payload) if payload else {}))
    except urllib.error.HTTPError as e:
        payload = e.read()
        try:    return e.code, json.loads(payload)
        except: return e.code, {"raw": payload[:400].decode(errors="replace")}

code, me = gh("GET", "/user")
if code != 200:
    sys.exit(f"❌ توکن معتبر نیست ({code}): {me.get('message')}")
print(f"✅ وارد شدی: {me['login']}")

code, repo = gh("GET", f"/repos/{REPO}")
if code != 200:
    sys.exit(f"❌ ریپو پیدا نشد: {REPO}")
print(f"✅ ریپو: {repo['full_name']}  (خالی: {repo['size'] == 0})")

scopes = ""
try:
    req = urllib.request.Request(f"{API}/user")
    req.add_header("Authorization", f"Bearer {TOKEN}")
    with urllib.request.urlopen(req) as r:
        scopes = r.headers.get("x-oauth-scopes", "")
except Exception:
    pass
if scopes:
    print(f"   دسترسی‌ها: {scopes}")
    if "workflow" not in scopes:
        print("⚠️  دسترسی `workflow` را نداری — آپلود .github شکست می‌خورد.")
        print("    برگرد و توکن را با تیک workflow بساز.")

In [ ]:
#@title گام ۳ — گرفتن سورس { display-mode: "form" }
#@markdown فایل `onvo-source.zip` را از گفتگو دانلود کن، بعد این سلول را اجرا
#@markdown کن و از پنجرهٔ آپلود همان فایل را انتخاب کن.

from google.colab import files
import shutil, pathlib

SRC = pathlib.Path("/content/onvo_src")
if SRC.exists(): shutil.rmtree(SRC)
SRC.mkdir(parents=True)

print("فایل onvo-source.zip را انتخاب کن…")
up = files.upload()

zname = next((n for n in up if n.lower().endswith(".zip")), None)
if not zname:
    sys.exit("❌ فایل zip پیدا نشد.")

with zipfile.ZipFile(io.BytesIO(up[zname])) as z:
    z.extractall(SRC)

# اگر همه‌چیز داخل یک پوشهٔ اضافه است، یک سطح بالا بیا
kids = [p for p in SRC.iterdir() if not p.name.startswith("__MACOSX")]
if len(kids) == 1 and kids[0].is_dir() and not (SRC / "settings.gradle.kts").exists():
    inner = kids[0]
    for item in inner.iterdir(): shutil.move(str(item), str(SRC / item.name))
    inner.rmdir()

all_files = [p for p in SRC.rglob("*") if p.is_file()]
print(f"\n✅ {len(all_files)} فایل استخراج شد")

wf = SRC / ".github/workflows/build.yml"
print(f"   build.yml موجود: {'بله ✅' if wf.exists() else 'خیر ❌'}")
if not wf.exists():
    sys.exit("❌ فایل ساخت پیدا نشد — zip ناقص است.")

In [ ]:
#@title گام ۴ — آپلود در ریپو { display-mode: "form" }
#@markdown همه‌چیز در **یک کامیت** با Git Data API فرستاده می‌شود،
#@markdown پس پوشهٔ `.github` هم درست منتقل می‌شود.

BRANCH = "main"

skip = {".git", "__MACOSX", ".DS_Store", "buildlogs", "local.properties"}
payload = []
for p in sorted(SRC.rglob("*")):
    if not p.is_file(): continue
    rel = p.relative_to(SRC)
    if any(part in skip for part in rel.parts): continue
    payload.append((str(rel).replace(os.sep, "/"), p))

print(f"در حال فرستادن {len(payload)} فایل…\n")

# blobs
tree = []
for i, (rel, path) in enumerate(payload, 1):
    content = path.read_bytes()
    st, blob = gh("POST", f"/repos/{REPO}/git/blobs", {
        "content": base64.b64encode(content).decode(),
        "encoding": "base64"
    })
    if st not in (200, 201):
        sys.exit(f"❌ خطا در {rel}: {blob}")
    # 100755 برای فایل‌های اجرایی
    mode = "100755" if rel in ("gradlew", "tools/setup-toolchain.sh") else "100644"
    tree.append({"path": rel, "mode": mode, "type": "blob", "sha": blob["sha"]})
    if i % 10 == 0 or i == len(payload):
        print(f"  {i}/{len(payload)}")

st, tr = gh("POST", f"/repos/{REPO}/git/trees", {"tree": tree})
if st not in (200, 201): sys.exit(f"❌ ساخت درخت: {tr}")

st, ref = gh("GET", f"/repos/{REPO}/git/ref/heads/{BRANCH}")
parents = [ref["object"]["sha"]] if st == 200 else []

st, commit = gh("POST", f"/repos/{REPO}/git/commits", {
    "message": "Onvo 0.3.0 — UI, network layer, on-device route brain",
    "tree": tr["sha"], "parents": parents
})
if st not in (200, 201): sys.exit(f"❌ کامیت: {commit}")

if parents:
    st, out = gh("PATCH", f"/repos/{REPO}/git/refs/heads/{BRANCH}",
                 {"sha": commit["sha"], "force": True})
else:
    st, out = gh("POST", f"/repos/{REPO}/git/refs",
                 {"ref": f"refs/heads/{BRANCH}", "sha": commit["sha"]})
if st not in (200, 201): sys.exit(f"❌ به‌روزرسانی شاخه: {out}")

print(f"\n✅ آپلود شد: https://github.com/{REPO}")

st, chk = gh("GET", f"/repos/{REPO}/contents/.github/workflows/build.yml")
print(f"   build.yml در ریپو: {'بله ✅' if st == 200 else 'خیر ❌'}")

In [ ]:
#@title گام ۵ — کلید امضا و secretها { display-mode: "form" }
#@markdown اگر قبلاً کلید ساخته‌ای و secretها در ریپو هستند، این سلول
#@markdown خودش تشخیص می‌دهد و **کاری نمی‌کند**. عوض کردن کلید یعنی
#@markdown کاربران فعلی دیگر نمی‌توانند آپدیت بگیرند.

FORCE_NEW_KEY = False  #@param {type:"boolean"}

import secrets, subprocess

st, existing = gh("GET", f"/repos/{REPO}/actions/secrets")
have = {s['name'] for s in existing.get('secrets', [])} if st == 200 else set()
needed = {"KEYSTORE_B64", "STORE_PASSWORD", "KEY_ALIAS", "KEY_PASSWORD"}

if needed.issubset(have) and not FORCE_NEW_KEY:
    print("✅ کلید امضا از قبل تنظیم شده — دست نمی‌زنم.")
    print("   secretهای موجود:", ", ".join(sorted(needed)))
    print("\n   اگر واقعاً می‌خواهی کلید جدید بسازی، تیک FORCE_NEW_KEY را")
    print("   بزن و دوباره اجرا کن. ولی بدان کاربران فعلی دیگر نمی‌توانند")
    print("   آپدیت بگیرند و باید اپ را پاک و دوباره نصب کنند.")
else:
    !pip -q install pynacl >/dev/null 2>&1
    from nacl import encoding, public

    STORE_PASS = secrets.token_urlsafe(24)
    ALIAS      = "onvo"
    KS         = "/content/onvo-release.jks"
    if os.path.exists(KS): os.remove(KS)

    r = subprocess.run([
        "keytool", "-genkeypair", "-v",
        "-keystore", KS, "-alias", ALIAS,
        "-keyalg", "RSA", "-keysize", "4096", "-validity", "10000",
        "-storepass", STORE_PASS, "-keypass", STORE_PASS,
        "-dname", "CN=Onvo, O=Onvo, C=DE"
    ], capture_output=True, text=True)
    if not os.path.exists(KS):
        sys.exit(f"❌ ساخت کلید ناموفق:\n{r.stderr}")
    print("✅ کلید ساخته شد")

    ks_b64 = base64.b64encode(open(KS, "rb").read()).decode()
    st, pk = gh("GET", f"/repos/{REPO}/actions/secrets/public-key")
    if st != 200: sys.exit(f"❌ کلید عمومی ریپو: {pk}")
    sealed = public.SealedBox(public.PublicKey(pk["key"].encode(), encoding.Base64Encoder()))

    def put_secret(name, value):
        enc = base64.b64encode(sealed.encrypt(value.encode())).decode()
        st, res = gh("PUT", f"/repos/{REPO}/actions/secrets/{name}",
                     {"encrypted_value": enc, "key_id": pk["key_id"]})
        print(f"   {name}: {'✅' if st in (201,204) else '❌ ' + str(res)}")

    print("\nدر حال تنظیم secretها…")
    for n, v in (("KEYSTORE_B64", ks_b64), ("STORE_PASSWORD", STORE_PASS),
                 ("KEY_ALIAS", ALIAS), ("KEY_PASSWORD", STORE_PASS)):
        put_secret(n, v)

    with open("/content/onvo-keystore-BACKUP.txt", "w") as f:
        f.write("Onvo signing key — KEEP THIS SAFE\n" + "="*50 + "\n\n")
        f.write(f"STORE_PASSWORD / KEY_PASSWORD:\n{STORE_PASS}\n\n")
        f.write(f"KEY_ALIAS:\n{ALIAS}\n\n")
        f.write("Losing this file means you can never ship an update to\n")
        f.write("existing users. Android rejects installs signed with a\n")
        f.write("different key.\n")

    print("\n📥 دو فایل را دانلود کن و جای امنی نگه دار:")
    files.download("/content/onvo-keystore-BACKUP.txt")
    files.download(KS)


In [ ]:
#@title گام ۶ — اجرای ساخت و انتظار { display-mode: "form" }
#@markdown حدود ۲۵ تا ۳۵ دقیقه طول می‌کشد. کامپایل Rust کند است.

st, _ = gh("POST", f"/repos/{REPO}/actions/workflows/build.yml/dispatches",
           {"ref": "main"})
print("اجرای دستی:", "✅" if st == 204 else f"قبلاً خودکار شروع شده ({st})")

time.sleep(12)
st, runs = gh("GET", f"/repos/{REPO}/actions/runs?per_page=1")
if st != 200 or not runs.get("workflow_runs"):
    sys.exit("❌ هیچ اجرایی پیدا نشد. تب Actions را دستی چک کن.")

run = runs["workflow_runs"][0]
RUN_ID = run["id"]
print(f"در حال دنبال کردن: {run['html_url']}\n")

start = time.time()
last  = None
while True:
    st, r = gh("GET", f"/repos/{REPO}/actions/runs/{RUN_ID}")
    if st != 200: print("خطای موقت API، تلاش دوباره…"); time.sleep(20); continue
    status, concl = r["status"], r["conclusion"]
    mins = int((time.time() - start) / 60)

    st2, jobs = gh("GET", f"/repos/{REPO}/actions/runs/{RUN_ID}/jobs")
    if st2 == 200:
        snap = " | ".join(
            f"{j['name'].split()[0]}:{j['conclusion'] or j['status']}"
            for j in jobs["jobs"]
        )
        if snap != last:
            print(f"[{mins:>3}د] {snap}")
            last = snap

    if status == "completed":
        print(f"\n{'✅ موفق' if concl == 'success' else '❌ ' + str(concl)}")
        break
    if mins > 60:
        print("\n⏱️ بیش از حد طول کشید. لینک بالا را دستی چک کن.")
        break
    time.sleep(30)

In [ ]:
#@title گام ۷ — دانلود APK { display-mode: "form" }

st, arts = gh("GET", f"/repos/{REPO}/actions/runs/{RUN_ID}/artifacts")
if st != 200 or not arts.get("artifacts"):
    sys.exit("❌ خروجی‌ای پیدا نشد. احتمالاً ساخت شکست خورده.")

apk_art = next((a for a in arts["artifacts"] if "apk" in a["name"].lower()),
               arts["artifacts"][0])
print(f"دانلود {apk_art['name']} ({apk_art['size_in_bytes']//1024} KB)…")

st, blob = gh("GET", apk_art["archive_download_url"], raw=True)
if st != 200: sys.exit(f"❌ دانلود ناموفق: {st}")

OUT = pathlib.Path("/content/apk"); OUT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    z.extractall(OUT)

apks = sorted(OUT.rglob("*.apk"))
print(f"\n✅ {len(apks)} فایل APK:\n")
for a in apks:
    print(f"   {a.name}  ({a.stat().st_size//1024//1024} MB)")

pick = next((a for a in apks if "universal" in a.name and "release" in str(a)),
     next((a for a in apks if "universal" in a.name),
          apks[0] if apks else None))

if pick:
    print(f"\n📥 دانلود: {pick.name}")
    files.download(str(pick))
else:
    sys.exit("❌ هیچ APK پیدا نشد.")

## گام ۸ — پاک کردن توکن

کار تمام شد. حالا توکن را باطل کن:

**[github.com/settings/tokens](https://github.com/settings/tokens)** ← کنار
`onvo-setup` ← **Delete**

secretهای ریپو مستقل از توکن‌اند و باقی می‌مانند، پس ساخت‌های بعدی
بدون مشکل کار می‌کنند.

---

### نصب

APK دانلودشده را باز کن. اگر اندروید هشدار داد، **Install anyway**.

اگر گوشیت **شیائومی** است، بعد از نصب حتماً به
**تنظیمات ← سازگاری با گوشی** برو و هر چهار مورد را فعال کن.

### اگر وصل نشد

**منو ← کنسول و لاگ** ← سوییچ «نمایش جزئیات فنی» را روشن کن ←
**کپی همه**. کلیدها و آی‌پی خودت موقع کپی حذف می‌شوند.